# UOT cross-corpus DFEW ↔ MAFW**Không cần GPU.** Chạy trên các file `.npz` mà baseline đã dump. Đặt accelerator = *None*để không tốn quota GPU.Toàn bộ 5 fold × 2 chiều mất khoảng **30 phút CPU**.

## 1. Clone repo

In [ ]:
WORK = "/kaggle/working/DEFR-UOT"import os, shutil# Bước ra ngoài TRƯỚC khi xoá. Chạy lại notebook lần hai thì cwd đang nằm trong# WORK (do %cd ở cuối cell), xoá thẳng sẽ huỷ chính thư mục tiến trình đang đứng:# getcwd() hỏng và mọi lệnh sau đó chết với "Unable to read current working directory".os.chdir("/kaggle/working")if os.path.exists(WORK):    shutil.rmtree(WORK)!git clone -b feat/uot-fusion --depth 1 https://github.com/YouttyLe-DSAI/DEFR-UOT.git {WORK}%cd {WORK}!git log --oneline -1

## 2. CONFIG`DUMPS` phải trỏ tới thư mục chứa các file `.npz` của baseline. Hai cách đưa lên:- **Add Input → Notebook Output**, chọn kernel baseline đã chạy, hoặc- Tải `Results/` về rồi upload thành một Kaggle Dataset

In [ ]:
DUMPS = "/kaggle/input/mma-baseline-dumps"   # sửa cho khớpEPS, TAU, ITERS = 0.05, 1.0, 200PRIOR  = "target-matched"     # xem Cell 5 để biết vì sao không dùng "uniform"METHOD = "av"                 # av | visual_only | audio_onlyimport globn = len(glob.glob(f"{DUMPS}/**/*.npz", recursive=True))print(f"tìm thấy {n} file .npz dưới {DUMPS}")

## 2b. Chưa có dumps? Tự sinh (tuỳ chọn — cần GPU)Bỏ qua nếu đã có dumps từ kernel baseline.Cần **20 file** (không phải 90): 4 tổ hợp checkpoint × tập test, mỗi tổ hợp 5 fold,chỉ phương thức `av`. Khoảng **~2 giờ GPU**.Điều kiện tiên quyết — **cả hai** corpus phải sẵn sàng trong phiên này, vì mỗicheckpoint được chạy trên **cả hai** tập test:```python# theo notebook UOT_PIPELINE.ipynb, chạy cho CẢ MAFW LẪN DFEW:!python tools/kaggle_setup.py --dataset MAFW --auto --out /kaggle/temp/data!python tools/retarget_annotations.py --dataset MAFW --new-root /kaggle/temp/data/mfaw/clips_faces --recount --drop-missing!python tools/kaggle_setup.py --dataset DFEW --auto --out /kaggle/temp/data!python tools/retarget_annotations.py --dataset DFEW --new-root /kaggle/temp/data/dfew/clip_224x224 --recount --drop-missing```Cộng 2 file encoder pretrain ở thư mục gốc repo (`GenerateModel.__init__` cần chúnglúc dựng model, dù `load_state_dict` sẽ ghi đè ngay sau đó).Đặt **Accelerator = GPU** cho phần này, rồi đổi lại **None** cho phần phân tích.

In [ ]:
CKPT_ROOT = "/kaggle/input/models/tunalmt/modelmma/pytorch/default/1/checkpoint"!cp {CKPT_ROOT}/pretrained.pth ./audiomae_pretrained.pth!cp {CKPT_ROOT}/mae_face_visualize_vit_base.pth ./mae_face_pretrain_vit_base.pth# Bỏ qua file đã tồn tại -> chạy lại được sau khi phiên bị ngắt!./tools/extract_all.sh {CKPT_ROOT} /kaggle/working/dumps av

Xong thì trỏ `DUMPS` ở Cell 2 sang `/kaggle/working/dumps` và chạy tiếp.Nên **Save Version** để giữ lại 20 file này — lần sau chỉ cần Add Input, khỏi tốn2 giờ GPU nữa.

## 3. Xem tên key trong dumpLoader tự dò `feature/features/feat/...` và `label/labels/target/...`. Nếu team dùngtên khác, cell này in ra để bạn truyền `--feat-key` / `--label-key`.

In [ ]:
import numpy as np, globfiles = sorted(glob.glob(f"{DUMPS}/**/*.npz", recursive=True))if not files:    print(f"Khong tim thay .npz nao duoi {DUMPS}")    print("  - Sua DUMPS o Cell 2 cho khop mount, hoac")    print("  - Chay Cell 2b de tu sinh 20 file (~2 gio GPU)")else:    z = np.load(files[0])    print(files[0], "\n")    for k in z.files:        print(f"  {k:<12} {z[k].shape}  {z[k].dtype}")    if "clf_weight" not in z.files:        print("\n  !! Thieu clf_weight -- dump nay sinh boi ban extract_features.py cu.")        print("     Barycentric projection can trong so classifier. Sinh lai bang Cell 2b.")

## 4. Một cặp + đường quét H1 (hình chính của bài)DFEW → MAFW: 4 lớp đích không có đối ứng ở nguồn.Cột `gap` là thứ H1 dự đoán: **≈ 0 tại 0% lớp lạ**, và tăng dần khi thêm mẫu lạ.

In [ ]:
SRC = f"{DUMPS}/baseline_B/dumps/dfew_dfew_fold1_{METHOD}.npz"TGT = f"{DUMPS}/baseline_A/dumps/dfew_mafw11_fold1_{METHOD}.npz"!python -m uot_crosscorpus.run --source {SRC} --target {TGT} \    --source-prior {PRIOR} --eps {EPS} --tau {TAU} --iters {ITERS} \    --sweep --out /kaggle/working/sweep_fold1.npz

## 5. Vì sao `--source-prior target-matched`Đề xuất dự đoán "ở 0% lớp lạ hai phương pháp phải trùng nhau". Điều đó **chỉ đúngkhi prior hai bên khớp**. Với biên đều, balanced OT khoá khối lượng mỗi lớp theo tầnsuất **nguồn** — mà báo cáo baseline đã ghi nhận DFEW và MAFW lệch prior rõ rệt.Đo trên dữ liệu tổng hợp cùng cấu trúc:| prior | gap tại **0% lớp lạ** | tại 100% ||---|---|---|| `uniform` | **+14.03 UAR** ← nhiễu prior | +10.91 || `target-matched` | **+0.00** ✅ | +0.16 |Với `uniform`, gần như toàn bộ gap đến từ lệch prior chứ không phải open-set — và sẽbị quy nhầm cho UOT. Chạy cell dưới để tự thấy trên dữ liệu thật.

In [ ]:
!python -m uot_crosscorpus.run --source {SRC} --target {TGT} \    --source-prior uniform --eps {EPS} --tau {TAU} --iters {ITERS} \    --sweep 2>&1 | tail -12

## 6. Toàn bộ 5 fold — chiều DFEW → MAFWLớp lạ nằm ở **đích** (open-set). Dùng cấu hình mặc định.

In [ ]:
!python -m uot_crosscorpus.batch --dumps-root {DUMPS} --direction dfew2mafw \    --method {METHOD} --source-prior {PRIOR} --eps {EPS} --tau {TAU} --iters {ITERS} \    --out /kaggle/working/uot_dfew2mafw.csv

## 7. Toàn bộ 5 fold — chiều MAFW → DFEW (partial)Ở chiều này 4 lớp thừa nằm ở **nguồn**, không phải đích. Bắt buộc dùng`--source-label-space full`: bỏ chúng đi là **xoá bài toán thay vì giải nó**, vàbalanced với unbalanced sẽ trùng nhau một cách tầm thường (đo được: gap +0.00 so với+5.14 khi giữ lại).

In [ ]:
!python -m uot_crosscorpus.batch --dumps-root {DUMPS} --direction mafw2dfew \    --method {METHOD} --source-label-space full \    --source-prior {PRIOR} --eps {EPS} --tau {TAU} --iters {ITERS} \    --out /kaggle/working/uot_mafw2dfew.csv

## 8. Quét lưới ε × τ

In [ ]:
import itertools, subprocessfor eps, tau in itertools.product([0.01, 0.05, 0.1], [0.1, 1.0, 10.0]):    print(f"===== eps={eps}  tau={tau} =====")    out = subprocess.run(        ["python", "-m", "uot_crosscorpus.batch", "--dumps-root", DUMPS,         "--direction", "dfew2mafw", "--method", METHOD, "--source-prior", PRIOR,         "--eps", str(eps), "--tau", str(tau), "--iters", str(ITERS)],        capture_output=True, text=True)    for line in out.stdout.splitlines():        if "balanced" in line or "paired t" in line:            print(" ", line.strip())

## 9. Kết quả

In [ ]:
import pandas as pd, globfor f in sorted(glob.glob("/kaggle/working/uot_*.csv")):    df = pd.read_csv(f)    print(f"\n{f}")    print(df.groupby("method")[["uar", "war", "auroc"]].agg(["mean", "std"]).round(3))

## Ghi chú- Nguồn và đích **bắt buộc cùng một checkpoint** — đặc trưng từ hai encoder khác nhau  không cùng metric space. Script cảnh báo nếu tiền tố tên file khác nhau.- Dump là từ tập **test**, nên "nguồn có nhãn" là DFEW test (~2341 mẫu/fold). Hợp lệ  vì đánh giá trên corpus **đích**, không rò rỉ.- `paired t` với n=5 chỉ mang tính chỉ báo, không phải kết luận thống kê chắc chắn.